# 背包问题

**类别：** 装箱

来源：[https://www.hexaly.com/templates/knapsack-problem](https://www.hexaly.com/templates/knapsack-problem)


## 问题

**背包问题**定义如下。考虑一组具有不同重量和价值的物品。我们需要选择一个物品子集放入一个已知容量的背包中。背包中物品的总重量不得超过其容量。目标是最大化背包中物品的总价值。

	

### 学到的建模原则

- 了解 Hexaly Optimizer 的建模风格：[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)


## 数据

我们提供来自 [OR Library](http://people.brunel.ac.uk/~mastjjb/jeb/info.html) 的实例。数据集的格式如下：

- 物品的数量
- 对每个物品，其重量
- 对每个物品，其价值
- 该实例的已知上界


## 模型

我们将背包问题建模为整数规划。对每个物品，我们定义一个 布尔决策变量，当物品被选中时为 1，否则为 0。我们使用 **sum** 算子计算被选中物品的总重量。注意，我们是从决策变量的值推导出这个值：总重量是一个中间表达式，而非决策变量。定义它之后，我们可以约束总重量小于背包的容量。类似地，我们定义另一个中间表达式，对应于被选中物品的总价值。最后，我们最大化该总价值。

尽管背包问题是 NP-hard 问题，但涉及数百万个物品的实例可以使用 Hexaly Optimizer 求解。


## Python 实现


In [1]:
from pathlib import Path

from optagent import OptModel, solve


def read_instance(filename):
    values = [int(value) for value in Path(filename).read_text(encoding="utf-8").split()]
    iterator = iter(values)
    nb_items = next(iterator)
    weights = [next(iterator) for _ in range(nb_items)]
    profits = [next(iterator) for _ in range(nb_items)]
    knapsack_capacity = next(iterator)
    return nb_items, weights, profits, knapsack_capacity


def main(input_file, output_file=None, time_limit=20):
    nb_items, weights, profits, knapsack_capacity = read_instance(input_file)
    model = OptModel()

    # x[i] is true when item i is selected.
    selected = [model.bool(name=f"item_{i}_selected") for i in range(nb_items)]

    # Keep the original Hexaly formulation: weight and value are intermediate expressions.
    knapsack_weight = model.sum(
        selected[i] * weights[i] for i in range(nb_items)
    )
    model.constraint(knapsack_weight <= knapsack_capacity, name="capacity")

    knapsack_value = model.sum(
        selected[i] * profits[i] for i in range(nb_items)
    )
    model.maximize(knapsack_value, name="knapsack_value")

    solution = solve(model, time_limit_s=float(time_limit))
    selected_items = [
        i for i, variable in enumerate(selected) if variable.value
    ]
    print(
        f"Items = {nb_items}; Capacity = {knapsack_capacity}; "
        f"Weight = {knapsack_weight.value}; Value = {knapsack_value.value}; "
        f"Status = {solution.feasible}"
    )
    print("Selected items:", selected_items)

    if output_file is not None:
        Path(output_file).write_text(
            f"{knapsack_value.value}\n"
            + " ".join(map(str, selected_items))
            + "\n",
            encoding="utf-8",
        )
    return solution


## 运行实例

以下代码格演示如何调用 OptAgent 背包模型。

In [2]:
INSTANCE_DIR = Path.cwd() / "instances"
print("Instances:", INSTANCE_DIR)


Instances: /Users/dongbox/work/opt-agent/examples/examples/hexaly/knapsack_problem/instances


In [3]:
solution_toy = main(INSTANCE_DIR / "toy.in", time_limit=1)


Starting OptAgent
Parameters: time_limit=1s
[   0.001s] initial feasible=true objective=[0]
[   0.001s] best #1 worker=0 feasible=true objective=[1]


Items = 8; Capacity = 102; Weight = 102; Value = 280; Status = True
Selected items: [2, 4, 5, 6, 7]


[   1.004s] best #10 worker=1 feasible=true objective=[280]
Solve summary:
  status: FEASIBLE
  objective: [280]
  improvements: 10
  evaluated: 2411
  wall_time: 1.00381s
  termination: deadline


In [4]:
solution_100_1 = main(INSTANCE_DIR / "kp_100_1.in", time_limit=1)


Starting OptAgent
Parameters: time_limit=1s
[   0.004s] initial feasible=true objective=[0]
[   0.010s] best #1 worker=0 feasible=true objective=[537]
[   0.213s] best #46 worker=1 feasible=true objective=[24182]
[   0.417s] best #56 worker=2 feasible=true objective=[26263]
[   0.633s] best #62 worker=2 feasible=true objective=[28248]
[   0.837s] best #68 worker=2 feasible=true objective=[29312]
[   1.011s] best #69 worker=2 feasible=true objective=[29464]
Solve summary:
  status: FEASIBLE
  objective: [29464]
  improvements: 69
  evaluated: 1021
  wall_time: 1.01068s
  termination: workers_done


Items = 100; Capacity = 24379; Weight = 24260; Value = 29464; Status = True
Selected items: [0, 2, 3, 4, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 18, 19, 22, 23, 24, 26, 27, 28, 30, 32, 35, 37, 41, 45, 49, 52, 55, 56, 58, 62, 65, 66, 70, 71, 73, 75, 76, 78, 82, 86, 88, 93, 94, 95, 96, 99]
